# 🚀 EchoVision: Cloud Multi-Modal Video QA Framework
### ⚡ Kaggle Notebook T4 GPU Edition (Multi-Model & Serverless API Enabled)

This notebook provides a complete environment for **EchoVision** with state-of-the-art multi-modal models:
- **LLM Generators (via Serverless API / Local)**: `qwen2.5-v1-72b-instruct`, `gemma-4-31b`, `phi-3.5-vision-instruct`
- **Visual Captioning Models**: `Salesforce/blip-image-captioning-base`, `Salesforce/blip-image-captioning-large`, `HuggingFaceTB/SmolVLM-256M-Instruct`, `wraps/moondream-caption`
- **Audio Embedding Model**: `FacebookAI/roberta-base` (768-dim normalized representations)
- **One-by-One Model Benchmarking**: Dedicated runner (`run_model_experiments.py`) to systematically test each model and compile comparison tables.

> **GPU Strategy**: Tuned for Kaggle **Tesla T4 (16 GB)** using memory-safe sequential extraction and remote API inference for heavy models (31B–72B).

## 1️⃣ Hardware & Environment Verification
Verify that Kaggle has assigned the Tesla T4 GPU and Internet connectivity is active.


In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print("[✓] Tesla T4 GPU confirmed!")
else:
    print("[!] Please turn on GPU in the Kaggle notebook settings panel on the right.")
print("=" * 60)


## 2️⃣ Configure API Keys (Hugging Face & OpenAI)
On Kaggle, add your `HF_TOKEN` in **Add-ons ➔ Secrets** for secure automated loading, or input it below.
This enables serverless API inference for `qwen2.5-v1-72b-instruct`, `gemma-4-31b`, and `phi-3.5-vision-instruct`.

In [ ]:
import os, getpass

# 1. Hugging Face API Token
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face Token (hf_..., press Enter to skip): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["USE_API"] = "1"
    print("[✓] Hugging Face Token configured! Serverless API inference active.")
    try:
        from huggingface_hub import HfApi
        user_info = HfApi(token=hf_token).whoami()
        print(f"[✓] Authenticated as Hugging Face user: {user_info.get('name', 'User')}")
    except Exception as e:
        print(f"[!] Token validation note: {e}")
else:
    print("[i] No Hugging Face Token provided. Falling back to local offline models.")

# 2. Optional OpenAI API Key
openai_key = None
try:
    from kaggle_secrets import UserSecretsClient
    openai_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception:
    pass

if not openai_key:
    openai_key = os.getenv("OPENAI_API_KEY")

if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    print("[✓] OpenAI API Key configured.")


## 3️⃣ Setup Workspace in `/kaggle/working` & Clone Repository
Clone [https://github.com/G-shubham18/online_EVL.git](https://github.com/G-shubham18/online_EVL.git) into `/kaggle/working` and set up the Python search path.


In [ ]:
import os, sys

WORKING_DIR = "/kaggle/working"
os.chdir(WORKING_DIR)

REPO_URL = "https://github.com/G-shubham18/online_EVL.git"
REPO_DIR = "online_EVL"

# Clone repository from GitHub into Kaggle working directory or pull latest updates
if not os.path.exists("main.py"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        !git clone {REPO_URL}
    os.chdir(os.path.join(WORKING_DIR, REPO_DIR))

print("Pulling latest framework code from GitHub...")
!git pull

sys.path.append(os.getcwd())
print("Working Directory:", os.getcwd())
!ls -lh


## 4️⃣ Install Multi-Modal Dependencies
Install `ffmpeg`, `libsndfile1`, and all required multi-modal packages:


In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q -r requirements.txt

import faster_whisper
import transformers
import scenedetect
import chromadb
import faiss
import rank_bm25
import timm
import openai
print("\n[✓] All dependencies installed successfully on Kaggle!")


## 5️⃣ Configure Model Parameters & T4 Memory Strategy
Select your active model combination:
- **LLM**: `qwen2.5-v1-72b-instruct` | `gemma-4-31b` | `phi-3.5-vision-instruct`
- **Captioning**: `Salesforce/blip-image-captioning-base` | `Salesforce/blip-image-captioning-large` | `HuggingFaceTB/SmolVLM-256M-Instruct` | `wraps/moondream-caption`
- **Audio Embedding**: `FacebookAI/roberta-base`
- **Sequential Ingestion**: Memory-safe extraction to avoid out-of-memory errors on Kaggle T4.

In [ ]:
import os

# Active Model Selection
os.environ["LLM_MODEL"] = "qwen2.5-v1-72b-instruct"           # qwen2.5-v1-72b-instruct | gemma-4-31b | phi-3.5-vision-instruct
os.environ["CAPTION_MODEL"] = "Salesforce/blip-image-captioning-base" # blip-base | blip-large | SmolVLM | moondream
os.environ["AUDIO_EMBED_MODEL"] = "FacebookAI/roberta-base"   # FacebookAI/roberta-base (768-dim)
os.environ["USE_API"] = "1"

# Feature Extraction Parameters
os.environ["WHISPER_MODEL"] = "large-v3-turbo"
os.environ["AST_MODEL"] = "MIT/ast-finetuned-audioset-10-10-0.4593"
os.environ["TEXT_EMBEDDING_MODEL"] = "BAAI/bge-large-en-v1.5"
os.environ["RTDETR_MODEL"] = "PekingU/rtdetr_r50vd"
os.environ["GROUNDING_DINO_MODEL"] = "IDEA-Research/grounding-dino-tiny"
os.environ["ENABLE_BM25"] = "1"
os.environ["BM25_WEIGHT"] = "0.4"
os.environ["CONCURRENT_INGESTION"] = "0"  # Memory-safe sequential mode for T4

import config
print("=" * 65)
print(f"Device                 : {config.DEVICE} (GPU: {config.IS_GPU})")
print(f"Active LLM Model       : {config.ACTIVE_LLM_MODEL}")
print(f"Active Caption Model   : {config.ACTIVE_CAPTION_MODEL}")
print(f"Active Audio Embed Model: {config.ACTIVE_AUDIO_EMBED_MODEL}")
print(f"API Inference Active   : {config.USE_API} (HF_TOKEN set: {bool(config.HF_TOKEN)})")
print(f"Ingestion Pipeline     : Sequential (Memory-Safe for Tesla T4)")
print("=" * 65)


## 6️⃣ Discover Videos & Inspect Questions Dataset
Scan `smoketest/videos` and load questions from `smoketest/json`.

> **Note**: Due to GitHub file-size limits, `.mp4` video files are gitignored. If `smoketest/videos/` is empty upon cloning, this cell will automatically generate a sample test video clip (`v_96vBhCFBbQk.mp4`) matching the dataset question via FFmpeg so you can run the complete pipeline immediately!


In [ ]:
import os, json
from ingestion import discover_videos

videos_dir = "smoketest/videos"
os.makedirs(videos_dir, exist_ok=True)
videos = discover_videos(videos_dir)

if not videos:
    print("[!] No videos found in smoketest/videos (videos are gitignored in GitHub repository).")
    print("[+] Generating a 5-second test video clip with audio tone via FFmpeg...")
    sample_clip = os.path.join(videos_dir, "v_96vBhCFBbQk.mp4")
    !ffmpeg -y -f lavfi -i testsrc=duration=5:size=640x360:rate=25 -f lavfi -i sine=frequency=440:duration=5 -c:v libx264 -c:a aac "{sample_clip}" -loglevel error
    videos = discover_videos(videos_dir)
    print(f"[✓] Created test video: {sample_clip}")

print(f"\nDiscovered {len(videos)} video(s) in {videos_dir}:")
for v in videos[:6]:
    print(f"  - {os.path.basename(v)}")

json_file = "smoketest/json/smoketest_questions.json"
if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    print(f"\nLoaded {len(q_data)} questions from {json_file}")
    print("Sample Question Entry:")
    print(json.dumps(q_data[1] if len(q_data) > 1 else q_data[0], indent=2))


## 7️⃣ Run Stage 1 (Sequential Extraction on T4 GPU)
Extracts timestamped speech, audio events, roberta-base audio embeddings, visual detections, and captions into isolated index stores.

In [ ]:
!python ingestion.py --dataset_dir smoketest --sequential \
    --caption_model "$CAPTION_MODEL" \
    --audio_embed_model "$AUDIO_EMBED_MODEL" \
    --use_api


## 8️⃣ Run Stage 2 & 3 (Decoupled Retrieval & Answer Generation)
Executes multi-modal decoupled retrieval, cross-encoder + BM25 re-ranking, and grounded generation with active LLM.

In [ ]:
!python main.py --dataset_dir smoketest --output_dir output \
    --llm_model "$LLM_MODEL" \
    --caption_model "$CAPTION_MODEL" \
    --audio_embed_model "$AUDIO_EMBED_MODEL" \
    --use_api


## 9️⃣ View Evaluation Metrics & Sample Predictions


In [ ]:
import os, json
import pandas as pd

eval_path = "output/evaluation_summary.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print("=" * 60)
    print("                 EVALUATION METRICS REPORT                  ")
    print("=" * 60)
    metrics = eval_data.get("metrics", {})
    print(f"Total Evaluated Questions    : {metrics.get('total_evaluated', 0)}")
    print(f"Exact Match Accuracy         : {metrics.get('exact_match_accuracy', 0)}%")
    print(f"Relaxed Substring Accuracy   : {metrics.get('relaxed_accuracy', 0)}%")
    if metrics.get("category_breakdown"):
        print("\nCategory Breakdown:")
        for cat, c_res in metrics["category_breakdown"].items():
            print(f"  - {cat:<20}: {c_res['accuracy']:>6.2f}% ({c_res['correct']}/{c_res['total']})")
    print("=" * 60)

pred_path = "output/smoketest_questions.json"
if os.path.exists(pred_path):
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    df = pd.DataFrame(preds)
    cols = [c for c in ["video_id", "category", "question", "predicted_answer", "ground_truth_answer"] if c in df.columns]
    display(df[cols].head(15))


## 🔟 Run Model Experiments One by One (`run_model_experiments.py`)
Benchmark each model **one by one** with isolated outputs and automatic evaluation reporting:
- **LLMs one by one**: `qwen2.5-v1-72b-instruct` ➔ `gemma-4-31b` ➔ `phi-3.5-vision-instruct`
- **Caption models one by one**: `blip-base` ➔ `blip-large` ➔ `SmolVLM` ➔ `moondream`
- **Audio embed model**: `FacebookAI/roberta-base`
- Results saved to `output/model_benchmarks/<model_name>/` and master summary compiled automatically.

In [ ]:
# Run all LLMs one by one via API
!python run_model_experiments.py --mode llm --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To test captioning models one by one, uncomment:
# !python run_model_experiments.py --mode caption --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To test audio embedding model, uncomment:
# !python run_model_experiments.py --mode audio --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To run all models sequentially in one run, uncomment:
# !python run_model_experiments.py --mode all --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest


In [ ]:
# Display Master Benchmark Markdown Report
import os
from IPython.display import Markdown, display

summary_path = "output/model_benchmarks/master_benchmark_summary.md"
if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("Benchmark summary not found yet. Run an experiment above to generate results.")


## 1️⃣1️⃣ Run Scientific Ablation Experiments (`run_ablations.py`)


In [ ]:
!python run_ablations.py --dataset_dir smoketest --output_dir output/ablations


## 1️⃣2️⃣ Interactive Single-Video Query Demo


In [ ]:
import os
from ingestion import discover_videos, Stage1Ingestor
from main import answer_question_for_video
from stage2_online.question_classifier import QuestionClassifier
from stage2_online.deduplicator import Deduplicator
from stage2_online.reranker import ReRanker
from stage2_online.sufficiency_gate import SufficiencyGate
from stage3_generator.generator import Generator

sample_videos = discover_videos("smoketest/videos")
if sample_videos:
    test_video = sample_videos[0]
    test_question = "what is behind the person in blue?"
    
    llm_choice = os.getenv("LLM_MODEL", "qwen2.5-v1-72b-instruct")
    caption_choice = os.getenv("CAPTION_MODEL", "Salesforce/blip-image-captioning-base")
    audio_choice = os.getenv("AUDIO_EMBED_MODEL", "FacebookAI/roberta-base")
    
    print(f"Selected Video: {test_video}")
    print(f"Question      : {test_question}")
    print(f"Active Models : LLM={llm_choice} | Caption={caption_choice} | Audio={audio_choice}\n")
    
    ingestor = Stage1Ingestor(
        caption_model=caption_choice,
        audio_embed_model=audio_choice,
        use_api=os.getenv("USE_API", "1").lower() in ("1", "true", "yes"),
        hf_token=os.getenv("HF_TOKEN")
    )
    indexer, reused, err = ingestor.process_single_video(test_video)
    
    if indexer:
        shared_components = {
            'qc': QuestionClassifier(),
            'dedup': Deduplicator(),
            'reranker': ReRanker(),
            'gate': SufficiencyGate(),
            'generator': Generator(
                model=llm_choice,
                use_api=os.getenv("USE_API", "1").lower() in ("1", "true", "yes"),
                hf_token=os.getenv("HF_TOKEN")
            )
        }
        answer = answer_question_for_video(indexer, test_question, shared_components)
        print("\n" + "=" * 50)
        print(f"PREDICTED ANSWER: {answer}")
        print("=" * 50)


## 1️⃣3️⃣ Export & Archive Output Results
Compress the output directory into a zip file available in the Kaggle Output viewer for direct download.


In [ ]:
import os, shutil

if os.path.exists("output"):
    shutil.make_archive("/kaggle/working/echovision_kaggle_results", 'zip', "output")
    print("[✓] Results archived to /kaggle/working/echovision_kaggle_results.zip")
    print("[i] Download it directly from the Kaggle Output section on the right sidebar.")
else:
    print("[!] Output directory 'output/' does not exist yet. Please run Stage 2 & 3 first.")
